**Anomaly detection in the S&P 500: eight labs with Python and Google Finance.** Data: Stock_Data_Template.xlsx. Run the cells from top to bottom.

**Setup.** Run this cell first. When the *Choose Files* button appears under it, choose **Stock_Data_Template.xlsx** (last close 19 September 2026).

In [2]:
# ===== Setup: run this cell first =====
import glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

found = glob.glob('*.xlsx')
if not found:              # Colab shows a "Choose Files" button
    from google.colab import files
    found = [n for n in files.upload() if n.endswith('.xlsx')]
raw = pd.read_excel(found[0], sheet_name='Stocks')

# keep US common shares priced at $1 or more
nc = (r'\b(?:warrants?|units?|rights?|notes?|preferred|depositary|'
      r'depository|acquisition corp|debentures?|subordinated|zones|'
      r'due \d{4})\b|\d+(?:\.\d+)?\s?%')
d = raw[(raw['Country'] == 'United States') & (raw['Last Sale'] >= 1)
        & ~raw['Name'].str.contains(nc, case=False, regex=True)].copy()
# one share class per company (GOOGL kept, GOOG dropped)
pat = r'\s+(Class [A-Z]|Common|Capital Stock|Ordinary).*$'
d['Company'] = d['Name'].str.replace(pat, '', regex=True).str.strip()
d = d.sort_values('Market Cap', ascending=False)
d = d.drop_duplicates('Company')
# history check: recent prices must fit inside the 1-year range
lo, hi = d['1Y Min'] * 0.9, d['1Y Max'] * 1.1
d['ok'] = d['Last Sale'].between(lo, hi)
for c in ['Price 1M Ago', 'Price 6M Ago']:
    d['ok'] &= d[c].isna() | d[c].between(lo, hi)
top = d[d['ok']].head(500).copy()        # S&P 500 proxy
print(len(raw), 'rows,', len(d), 'US common stocks,',
      (~d['ok']).sum(), 'with doubtful history,', len(top), 'in top 500')

Saving Stock Data Template.xlsx to Stock Data Template.xlsx
7033 rows, 3933 US common stocks, 19 with doubtful history, 500 in top 500


**Lab 1 (Part 1, after the feature-kit slide).** The trader's feature kit for the 500 largest companies. *Google Finance:* open NVDA and compare Year range and Market cap with the printed row.

**Terms:** `move` price change on the day, % · `turnover` value traded as % of market cap · `r1m`, `r6m`, `r1y` return over 1 month, 6 months, 1 year, % · `range_pos` where the price sits between the 1-year low (0) and high (100) · `vs_avg` price compared with its 1-year average, % (+13 means 13% above the average) · `1% are below` only 5 of 500 companies are lower · `median` the middle company · `99% are below` only 5 of 500 are higher.

In [3]:
# ===== Wed lab 1: the trader's feature kit from the Excel file =====
K = top.copy()
K['move'] = 100 * K['% Change']                     # today, %
K['turnover'] = 100 * K['Volume'] * K['Last Sale'] / K['Market Cap']
K['r1m'] = 100 * (K['Last Sale'] / K['Price 1M Ago'] - 1)
K['r6m'] = 100 * (K['Last Sale'] / K['Price 6M Ago'] - 1)
K['r1y'] = 100 * (K['Last Sale'] / K['Price 1Y Ago'] - 1)
rng = K['1Y Max'] - K['1Y Min']
K['range_pos'] = 100 * (K['Last Sale'] - K['1Y Min']) / rng
K['vs_avg'] = 100 * (K['Last Sale'] / K['1Y Avg'] - 1)
F = ['move', 'turnover', 'r1m', 'r6m', 'r1y', 'range_pos', 'vs_avg']
q = K[F].quantile([0.01, 0.5, 0.99]).T
q.columns = ['1% are below', 'median', '99% are below']
print(q.round(1))
# compare this row with the NVDA page on google.com/finance
cols = ['Last Sale', '1Y Min', '1Y Max', 'Market Cap'] + F
row = K.set_index('Symbol').loc['NVDA', cols]
print(row.apply(lambda v: f'{v:,.2f}'))      # readable numbers

           1% are below  median  99% are below
move               -5.1    -0.5            7.2
turnover            0.4     1.6           12.0
r1m               -21.5    -4.6           24.1
r6m               -33.0     6.8          152.1
r1y               -52.1     9.7          335.0
range_pos           0.1    51.8           98.7
vs_avg            -31.7     1.8           77.7
Last Sale                   222.27
1Y Min                      165.17
1Y Max                      235.74
Market Cap    5,356,707,000,000.00
move                          1.34
turnover                      0.79
r1m                           4.33
r6m                          24.40
r1y                          25.09
range_pos                    80.91
vs_avg                       13.03
Name: NVDA, dtype: object


**Lab 2 (Part 1, after 'Reading the score').** Isolation Forest scores, WATCH and ALERT, average depth. *Google Finance:* make a watchlist 'ALERT' with the five ALERT names and read their news.

**Terms:** `s` anomaly score from 0 to 1, higher = more unusual · `depth` average number of cuts needed to isolate the company, fewer = more unusual · `level` WATCH = top 5% of scores, ALERT = top 1%, empty = normal.

In [4]:
# ===== Wed lab 2: score the 500 and set WATCH and ALERT =====
X = K[F].fillna(K[F].median())
forest = IsolationForest(n_estimators=300, max_samples=256,
                         random_state=0).fit(X)
K['s'] = -forest.score_samples(X)                   # s(x) in (0, 1)
watch, alert = K['s'].quantile([0.95, 0.99])
K['level'] = np.where(K['s'] >= alert, 'ALERT',
                      np.where(K['s'] >= watch, 'WATCH', ''))
psi = 256
c = 2 * (np.log(psi - 1) + 0.5772) - 2 * (psi - 1) / psi
K['depth'] = -c * np.log2(K['s'])                   # average path E[h]
print(f'c(psi) = {c:.2f}  WATCH s >= {watch:.3f}  ALERT s >= {alert:.3f}')
print(K.groupby('level')['depth'].median().round(2))
cols = ['Symbol', 's', 'depth', 'level', 'move', 'r1m', 'turnover']
print(K.nlargest(10, 's')[cols].round(2).to_string(index=False))

c(psi) = 10.24  WATCH s >= 0.553  ALERT s >= 0.671
level
         14.09
ALERT     5.34
WATCH     8.23
Name: depth, dtype: float64
Symbol    s  depth level  move   r1m  turnover
  SNDK 0.78   3.75 ALERT 10.99 21.01     12.15
  MRNA 0.70   5.31 ALERT -2.55 -3.02      5.07
  DELL 0.70   5.34 ALERT -3.46 25.82      1.85
    BE 0.69   5.38 ALERT -5.39 22.16     24.95
  MSTR 0.69   5.59 ALERT 16.39 21.36     14.23
  SMTC 0.67   5.90 WATCH  3.82 45.08      6.51
    MU 0.65   6.34 WATCH  3.92  8.88      3.17
  LITE 0.64   6.53 WATCH  4.17  5.12     10.33
  ILMN 0.64   6.60 WATCH -2.27  6.50     21.58
  OKTA 0.61   7.26 WATCH -4.03 39.63      3.28


**Lab 3 (Part 1, after Situation 3).** Data rule and forest side by side. *Google Finance:* open BH, CAT and BHE and compare Year range with the file.

**Terms:** `rule` True if the price history fails the check · `forest` True if the forest puts the row in its top 5% · `rng` width of the 1-year range, ln(Max/Min).

In [5]:
# ===== Wed lab 3: catch bad data before you trust a flag =====
A = d[d['Market Cap'] >= 3e8].copy()
A['r1m'] = np.log(A['Last Sale'] / A['Price 1M Ago'])
A['r1y'] = np.log(A['Last Sale'] / A['Price 1Y Ago'])
A['rng'] = np.log(A['1Y Max'] / A['1Y Min'])
XA = A[['r1m', 'r1y', 'rng']].replace([np.inf, -np.inf], np.nan)
XA = XA.fillna(XA.median())
f3 = IsolationForest(n_estimators=300, random_state=0).fit(XA)
A['s'] = -f3.score_samples(XA)
A['forest'] = A['s'] >= A['s'].quantile(0.95)       # top 5%
A['rule'] = ~A['ok']                                # setup-cell rule
print(pd.crosstab(A['rule'], A['forest']))
cols = ['Symbol', 'Last Sale', 'Price 1M Ago', '1Y Min', '1Y Max',
        'rule', 'forest']
print(A[A['rule'] | A['Symbol'].isin(['BH'])][cols].to_string(index=False))

forest  False  True 
rule                
False    2521    122
True        2     11
Symbol  Last Sale  Price 1M Ago  1Y Min   1Y Max  rule  forest
   CAT     808.99           NaN  0.0352   0.0352  True   False
  CASY     597.48        1.7400  1.1200  11.7000  True    True
  CART      45.37        5.2800  4.1500   6.7700  True    True
   CAR     117.71       24.4301 23.4500  25.1500  True    True
  CARG      33.21        6.8000  4.4800  10.5400  True    True
   BHF      49.95        2.4300  1.1200   2.5100  True    True
   BHE      75.40        1.4400  0.6150   2.4600  True    True
  CASH      76.17       16.7500 11.0500  18.6200  True    True
  AVLN      31.52           NaN 23.4000  23.4000  True   False
    BH     345.34        6.5900  1.0500 792.0000 False    True
  CASS      54.74        5.7700  3.7700   8.4000  True    True
  CARS      11.14        0.4700  0.3590   3.3700  True    True
  CAPR       9.06        0.5700  0.3630   1.0600  True    True
  CARL      15.58        9.8100  7

**Lab 4 (Part 1, after Situation 4).** Sector part and company part of today's move. *Google Finance:* open MSTR, Compare with BTC-USD and the Nasdaq Composite.

**Terms:** `sector_move` median move of the stock's sector that day, % · `own` move minus sector_move, the company's own part · `turn` turnover on a log scale.

In [6]:
# ===== Wed lab 4: is the move about the company or the sector? =====
S = d[d['ok'] & (d['Market Cap'] >= 1e9)].copy()
S['move'] = 100 * S['% Change']
S['sector_move'] = S.groupby('Sector')['move'].transform('median')
S['own'] = S['move'] - S['sector_move']             # company part
S['turn'] = np.log10(100 * S['Volume'] * S['Last Sale']
                     / S['Market Cap'])
XS = S[['move', 'own', 'turn']].fillna(S[['move', 'own', 'turn']].median())
f4 = IsolationForest(n_estimators=300, random_state=0).fit(XS)
S['s'] = -f4.score_samples(XS)
print(S.groupby('Sector')['move'].median().round(2).sort_values())
cols = ['Symbol', 'Sector', 'move', 'sector_move', 'own', 's']
print(S.nlargest(10, 's')[cols].round(2).to_string(index=False))

Sector
Consumer Staples         -1.13
Utilities                -1.07
Basic Materials          -0.83
Energy                   -0.78
Technology               -0.77
Real Estate              -0.76
Health Care              -0.71
Consumer Discretionary   -0.68
Telecommunications       -0.53
Miscellaneous            -0.46
Industrials              -0.34
Finance                  -0.11
Name: move, dtype: float64
Symbol      Sector  move  sector_move   own    s
  MARA     Finance 13.75        -0.11 13.86 0.78
  MSTR  Technology 16.39        -0.77 17.16 0.78
  PURR     Finance  9.82        -0.11  9.93 0.76
  SBET  Technology 13.20        -0.77 13.97 0.75
  SNDK  Technology 10.99        -0.77 11.76 0.75
  COIN     Finance 11.66        -0.11 11.77 0.74
  GLXY     Finance 10.82        -0.11 10.93 0.73
   SMR Industrials -8.52        -0.34 -8.18 0.73
  BMNR     Finance  8.79        -0.11  8.90 0.72
  CLSK     Finance  8.39        -0.11  8.50 0.71


**Lab 5 (Part 2, after Situation 8).** The move in units of the stock's own volatility. *Google Finance:* 1-year charts of WOR and AXTI.

**Terms:** `sig_d` the stock's normal daily move, %, from its 1-year high and low · `z` move divided by sig_d: how many normal days the move is worth.

In [7]:
# ===== Wed lab 5: the same move in a calm and in a stormy stock =====
V = d[d['ok'] & (d['Market Cap'] >= 1e9)].copy()
V['move'] = 100 * V['% Change']
sig_y = np.log(V['1Y Max'] / V['1Y Min']) / np.sqrt(8 / np.pi)
V['sig_d'] = 100 * sig_y / np.sqrt(252)             # daily vol, %
V['z'] = V['move'] / V['sig_d']                     # own-vol units
XV = V[['move', 'z']].fillna(0)
f5 = IsolationForest(n_estimators=300, random_state=0).fit(XV)
V['s'] = -f5.score_samples(XV)
V['level'] = np.where(V['s'] >= V['s'].quantile(0.99), 'ALERT',
                      np.where(V['s'] >= V['s'].quantile(0.95),
                               'WATCH', ''))
band = V[V['move'].abs().between(3, 5)]             # similar raw moves
cols = ['Symbol', 'move', 'sig_d', 'z', 'level']
print(band.reindex(band['z'].abs().sort_values().index)[cols]
      .round(2).iloc[[0, 1, 2, -3, -2, -1]].to_string(index=False))

Symbol  move  sig_d     z level
  AXTI  3.36  14.03  0.24      
  ERAS -3.72   9.94 -0.37      
  AEHR  3.25   7.96  0.41      
   RRR -3.56   1.15 -3.10 WATCH
  ADIG -4.34   1.34 -3.24 WATCH
   WOR -4.27   1.13 -3.80 WATCH


**Lab 6 (Part 2, after Situation 11).** Two portfolios for Google Finance. *Google Finance:* Portfolios, New portfolio, Add investment with date 19 Sep 2026.

**Terms:** `budget` $1,000 per stock · `shares` budget divided by the 19 Sep price · `flagged` ten highest scores · `control` ten random normal companies.

In [8]:
# ===== Wed lab 6: ALERT portfolio vs control in Google Finance =====
budget, day = 1000, '19 Sep 2026'                  # last close in file
flagged = K.nlargest(10, 's')
control = K[K['level'] == ''].sample(10, random_state=1)
for name, p in [('ALERT', flagged), ('Control', control)]:
    p = p.assign(shares=(budget / p['Last Sale']).round(3))
    print(f'\n{name} portfolio: Add investment, date {day}')
    print(p[['Symbol', 'Last Sale', 'shares']].to_string(index=False))


ALERT portfolio: Add investment, date 19 Sep 2026
Symbol  Last Sale  shares
  SNDK    1791.82   0.558
  MRNA     154.04   6.492
  DELL     568.06   1.760
    BE     265.63   3.765
  MSTR     153.92   6.497
  SMTC     185.00   5.405
    MU    1015.80   0.984
  LITE     930.91   1.074
  ILMN     239.62   4.173
  OKTA     182.37   5.483

Control portfolio: Add investment, date 19 Sep 2026
Symbol  Last Sale  shares
   TDG    1086.05   0.921
   MLM     490.64   2.038
    MS     202.58   4.936
     J     143.40   6.974
  ARES     123.80   8.078
   ADM      85.18  11.740
   URI    1013.91   0.986
  BIIB     215.50   4.640
  CTAS     197.64   5.060
   NSC     314.16   3.183


**Lab 7 (Part 2, after the routine slide).** A live morning list. *Google Sheets:* File, Import, Upload, morning_list.csv.

**Terms:** `Price now` live price · `Today %` live change on the day · `Vol / avg` today's volume divided by average volume, above 2 = twice the usual · `Since file %` change since the 19 Sep close.

In [ ]:
# ===== Wed lab 7: a live morning list in Google Sheets =====
M = K.nlargest(10, 's')[['Symbol', 'Last Sale', 'level', 's']].copy()
M['Symbol'] = M['Symbol'].str.replace('/', '.')     # BRK/B -> BRK.B
M['s'] = M['s'].round(3)
r = range(2, len(M) + 2)                            # sheet rows
M['Price now'] = [f'=GOOGLEFINANCE(A{i},"price")' for i in r]
M['Today %'] = [f'=GOOGLEFINANCE(A{i},"changepct")' for i in r]
M['Vol / avg'] = [f'=GOOGLEFINANCE(A{i},"volume")'
                  f'/GOOGLEFINANCE(A{i},"volumeavg")' for i in r]
M['Since file %'] = [f'=100*(E{i}/B{i}-1)' for i in r]
M.to_csv('morning_list.csv', index=False)
print(M.iloc[:, :4].to_string(index=False))
try:
    from google.colab import files
    files.download('morning_list.csv')              # then File > Import
except ImportError:
    print('saved morning_list.csv')

Symbol  Last Sale level     s
  SNDK    1791.82 ALERT 0.776
  MRNA     154.04 ALERT 0.698
  DELL     568.06 ALERT 0.697
    BE     265.63 ALERT 0.695
  MSTR     153.92 ALERT 0.685
  SMTC     185.00 WATCH 0.671
    MU    1015.80 WATCH 0.651
  LITE     930.91 WATCH 0.643
  ILMN     239.62 WATCH 0.640
  OKTA     182.37 WATCH 0.612


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Lab 8 (Part 2, after 'Correct and incorrect use').** Did flags from six months ago predict risk or direction?

**Terms:** `r_half` return in the half-year before the flag · `r_long` yearly trend of the 4 years before · `flag` True for the top 10% of scores six months ago · `next` return in the following 6 months · `abs` its size without the sign · `t direction`, `t risk` whether the difference is larger than luck (above 2 in size = yes).

In [ ]:
# ===== Wed lab 8: did old flags predict risk or direction? =====
need = ['Price 5Y Ago', 'Price 1Y Ago', 'Price 6M Ago']
B = top.dropna(subset=need).copy()
XB = pd.DataFrame({                                 # known 6M ago
    'r_half': np.log(B['Price 6M Ago'] / B['Price 1Y Ago']),
    'r_long': np.log(B['Price 1Y Ago'] / B['Price 5Y Ago']) / 4})
f8 = IsolationForest(n_estimators=300, random_state=0).fit(XB)
B['s'] = -f8.score_samples(XB)
B['flag'] = B['s'] >= B['s'].quantile(0.9)          # top 10%, 6M ago
B['next'] = 100 * np.log(B['Last Sale'] / B['Price 6M Ago'])
B['abs'] = B['next'].abs()
print(B.groupby('flag')[['next', 'abs']].agg(['size', 'mean']).round(1))
def t(col):                                         # flagged vs rest
    a, b = B.loc[B['flag'], col], B.loc[~B['flag'], col]
    return (a.mean() - b.mean()) / np.sqrt(a.var() / len(a)
                                           + b.var() / len(b))
print(f"t direction = {t('next'):.2f}   t risk = {t('abs'):.2f}")

      next        abs      
      size  mean size  mean
flag                       
False  425   7.7  425  16.6
True    48  24.1   48  33.5
t direction = 3.11   t risk = 4.24
